In [68]:
%pip install -Uqq fastai gradio nbdev

Note: you may need to restart the kernel to use updated packages.


In [69]:
from fastai.vision.all import load_learner
import gradio as gr

In [70]:
from fastai.vision.all import *

In [71]:
import pathlib

In [103]:
import torch
import torchvision.transforms as T
from PIL import Image

In [72]:
pathlib.PosixPath = pathlib.WindowsPath

In [73]:
#from google.colab import drive
#drive.mount('/content/drive')

In [74]:
#%cd /content/drive/MyDrive/Capstone_Project

In [76]:
trained_model = load_learner("model/safety_densenet121_model_Version_1.pkl")

In [109]:
categories = [
"Bangladesh Cracked or Damaged Asphalt Road",
"Bangladesh Crowded Pedestrian Footpath",
"Bangladesh Flooded Road",
"Bangladesh Pedestrians Crossing Busy Roads",
"Bangladesh Roads Potholes",
"Bus Stopping in Road",
"CNG-Autorickshaw in Traffic",
"Illegal Parking on Roadside in Bangladesh",
"Overloaded Rickshaw or Van"
]

print(f"Length of categories : {len(categories)}")

Length of categories : 9


In [110]:
def road_condition(image):
    img = Image.open(image).convert("RGB") if isinstance(image, str) else image

    tfms = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
    ])

    x = tfms(img).unsqueeze(0)
    trained_model.model.eval()

    with torch.no_grad():
        probs = torch.softmax(trained_model.model(x), dim=-1)[0]

    return dict(zip(categories, map(float, probs)))

In [111]:
#img = PILImage.create(f'test_images/image_2.jpg')
#img.to_thumb(128, 128)
#img

In [115]:
# Test it
road_condition("test_images/image_2.jpg")

{'Bangladesh Cracked or Damaged Asphalt Road': 0.00011356258619343862,
 'Bangladesh Crowded Pedestrian Footpath': 0.0005828966968692839,
 'Bangladesh Flooded Road': 0.002713157795369625,
 'Bangladesh Pedestrians Crossing Busy Roads': 0.029891522601246834,
 'Bangladesh Roads Potholes': 1.1921061741304584e-05,
 'Bus Stopping in Road': 0.016411062330007553,
 'CNG-Autorickshaw in Traffic': 0.007866025902330875,
 'Illegal Parking on Roadside in Bangladesh': 0.9383602142333984,
 'Overloaded Rickshaw or Van': 0.004049668554216623}

In [116]:
category_image = gr.Image()
category_label = gr.Label()
examples = [
    'test_images/image_1.jpg',
    'test_images/image_2.jpg'
]

In [117]:
with gr.Blocks() as iface:
    #gr.Markdown("### Image Classifier")
    category_image = gr.Image(type="pil")
    category_label = gr.Label()
    btn = gr.Button("Predict")

    btn.click(fn=road_condition, inputs=category_image, outputs=category_label)

    gr.Examples(
        examples=[
            'test_images/image_1.jpg',
            'test_images/image_2.jpg'
        ],
        inputs=category_image
    )

iface.launch(inline=False, share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://3d0020dba91db36b3f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
